In [8]:
print("hello")

hello


In [ ]:
import cv2 as cv
import numpy as np
import pyautogui as pag
import mediapipe as mp
import time

In [10]:

import threading

import cv2


class Camera(object):
    """
    #Base Camera object
    """

    def __init__(self):
        self._cam = None
        self._frame = None
        self._frame_width = None
        self._frame_height = None
        self._ret = False

        self.auto_undistortion = False
        self._camera_matrix = None
        self._distortion_coefficients = None

        self._is_running = False

    def _init_camera(self):
        """
        #This is the first for creating our camera
        #We should override this!
        """

        pass

    def start_camera(self):
        """
       # Start the running of the camera, without this we can't capture frames
       # Camera runs on a separate thread so we can reach a higher FPS
        """

        self._init_camera()
        self._is_running = True
        threading.Thread(target=self._update_camera, args=()).start()

    def _read_from_camera(self):
        """
       # This method is responsible for grabbing frames from the camera
       # We should override this!
        """

        if self._cam is None:
            raise Exception("Camera is not started!")

    def _update_camera(self):
        """
        #Grabs the frames from the camera
        """

        while True:
            if self._is_running:
                self._ret, self._frame = self._read_from_camera()
            else:
                break

    def get_frame_width_and_height(self):
        """
        #Returns the width and height of the grabbed images
        #:return (int int): width and height
        """

        return self._frame_width, self._frame_height

    def read(self):
        """
        #With this you can grab the last frame from the camera
        #:return (boolean, np.array): return value and frame
        """
        return self._ret, self._frame

    def release_camera(self):
        """
        #Stop the camera
        """

        self._is_running = False

    def is_running(self):
        return self._is_running

    def set_calibration_matrices(self, camera_matrix, distortion_coefficients):
        self._camera_matrix = camera_matrix
        self._distortion_coefficients = distortion_coefficients

    def activate_auto_undistortion(self):
        self.auto_undistortion = True

    def deactivate_auto_undistortion(self):
        self.auto_undistortion = False

    def _undistort_image(self, image):
        if self._camera_matrix is None or self._distortion_coefficients is None:
            import warnings
            warnings.warn("Undistortion has no effect because <camera_matrix>/<distortion_coefficients> is None!")
            return image

        h, w = image.shape[:2]
        new_camera_matrix, roi = cv2.getOptimalNewCameraMatrix(self._camera_matrix,
                                                               self._distortion_coefficients, (w, h),
                                                               1,
                                                               (w, h))
        undistorted = cv2.undistort(image, self._camera_matrix, self._distortion_coefficients, None,
                                    new_camera_matrix)
        return undistorted


class WebCamera(Camera):
    """
    #Simple Webcamera
    """

    def __init__(self, video_src=0):
        """
        #:param video_src (int): camera source code (it should be 0 or 1, or the filename)
        """

        super().__init__()
        self._video_src = video_src

    def _init_camera(self):
        super()._init_camera()
        self._cam = cv2.VideoCapture(self._video_src)
        self._ret, self._frame = self._cam.read()
        if not self._ret:
            raise Exception("No camera feed")
        self._frame_height, self._frame_width, c = self._frame.shape
        return self._ret

    def _read_from_camera(self):
        super()._read_from_camera()
        self._ret, self._frame = self._cam.read()
        if self._ret:
            if self.auto_undistortion:
                self._frame = self._undistort_image(self._frame)
            return True, self._frame
        else:
            return False, None

    def release_camera(self):
        super().release_camera()
        self._cam.release()


In [11]:
def nooffingers(hands, img):
    results = hands.process(img)
    index, middle, ring, pinky, thumb = 0,0,0,0,0
    indexco, middleco, ringco, pinkyco, thumbco = 0,0,0,0,0
    if results.multi_hand_landmarks:
        for handlms in results.multi_hand_landmarks:
            for id, lm in enumerate(handlms.landmark):
                if id==0:
                    wristpoint = lm
                    
                if id==2:
                    thumbref = lm
                    
                if id==4:
                    if abs(lm.x-wristpoint.x) > abs(thumbref.x-wristpoint.x):
                        thumb = 1
                        thumbco = lm
                        
                if id==8:
                    if lm.y<prev.y:
                        index = 1
                        indexco = lm
                        
                if id==12:
                    if lm.y<prev.y:
                        middle = 1
                        middleco = lm
                        
                if id==16:
                    if lm.y<prev.y:
                        ring = 1
                        ringco = lm
                        
                if id==20:
                    if lm.y<prev.y:
                        pinky = 1
                        pinkyco = lm
                        
                prev = lm
                
    return index, middle, ring, pinky, thumb, indexco, middleco, ringco, pinkyco, thumbco, results.multi_hand_landmarks

In [ ]:
#initiating video capture

webcam = cv.VideoCapture(0)

# Example usage:

#webcam = WebCamera(video_src=0)
#webcam.start_camera()

#for fps
prev_time = 0 
new_time = 0

#using hands method from mediapipe
hands = mpHands.Hands(static_image_mode = False, max_num_hands= 1, min_detection_confidence = 0.5 , min_tracking_confidence = 0.5)
mpDraw = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

plocX, plocY = 0,0
smoothening = 7
camwidth, camheight = 640, 480
screenwidth, screenheight = 1920, 1080
frameReduction = 100
pag.FAILSAFE=False
pag.PAUSE = 0.01

count20= 0
#display resolution
xd, yd = pag.size()
#capturing the frames for processing
while True:
        
    _, capture = webcam.read()
    
    #for the fps of the camera
    new_time =time.time()
    fps = 1/(new_time-prev_time)
    
    #The frame with hand
    frame = capture.copy()
    rgbframe = cv.cvtColor(frame,cv.COLOR_BGR2RGB)
    
    #detecting hands in the handframe
    index, middle, ring, pinky, thumb, indexco, middleco, ringco, pinkyco, thumbco, handLms =  nooffingers(hands, rgbframe)

    #for cursor control
    if index == 1:
        
        #for working only in roi
        if indexco.x>=0.15625 and indexco.x<=0.84375 and indexco.y>=0.2083 and indexco.y<=0.7916:
            
            #convert the coorinates into roi 
            x3 = (indexco.x-0.15625)/0.6875 
            y3 = (indexco.y-0.2083)/0.5833 
        
            clocX = plocX + (x3 - plocX) / smoothening
            clocY = plocY + (y3 - plocY) / smoothening
            
            print("{}:: X:{}, Y:{}, screenX:{}, screenY:{}".format(str(count20),str(clocX),str(clocY),str(clocX*screenwidth),str(clocY*screenheight)))
            pag.moveTo(clocX*screenwidth,clocY*screenheight)
        
            plocX = clocX
            plocY = clocY
            count20= 0
            
    #for mousedrags
    if index == 1 and middle == 1 and ring == 1 and pinky == 1 and thumb == 1:
        pag.mouseDown(button="left")
        while middle!=0 and ring!=0 and pinky!=0 and thumb!=0:
            _, capturedown = webcam.read()
            rgbframe = cv.cvtColor(capturedown,cv.COLOR_BGR2RGB)
            index, middle, ring, pinky, thumb, indexcor, middlecor, ringcor, pinkycor, thumbcor, handLms =  nooffingers(hands, rgbframe)
            if index == 1:
        
                #for working only in roi
                if indexcor.x>=0.15625 and indexcor.x<=0.84375 and indexcor.y>=0.2083 and indexcor.y<=0.7916:

                    #convert the coorinates into roi 
                    x3 = (indexcor.x-0.15625)/0.6875 
                    y3 = (indexcor.y-0.2083)/0.5833 

                    clocX = plocX + (x3 - plocX) / smoothening
                    clocY = plocY + (y3 - plocY) / smoothening

                    pag.moveTo(clocX*screenwidth, clocY*screenheight)

                    plocX = clocX
                    plocY = clocY
            #if middle==0 and ring==0 and pinky==0 and thumb==0:
            cv.rectangle(capturedown, (frameReduction, frameReduction), (camwidth - frameReduction, camheight - frameReduction),(255, 0, 255), 2)
            if handLms:
                for i in handLms:
                    mpDraw.draw_landmarks(capturedown, i, mpHands.HAND_CONNECTIONS,mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())
            cv.imshow("frame",capturedown)
            cv.waitKey(1)
            if middle == 0 and ring == 0 and pinky == 0 and thumb==0:
                cv.destroyAllWindows()
        pag.mouseUp(button="left", x=clocX, y=clocY)
                    
    #for scrolling up and down
    if index == 1 and middle == 1 and ring == 1 and pinky == 1:
        reference = middleco.y
        print(reference)
        cv.destroyAllWindows()
        while middle!=0 and ring!=0 and pinky!=0:
            #waits until the fingers other than index are removed, 
            #if not scrolling takes place based on the position of the middle finger relative to its reference position
            _, capturedown = webcam.read()
            rgbframe = cv.cvtColor(capturedown,cv.COLOR_BGR2RGB)
            index, middle, ring, pinky, thumb, indexcor, middlecor, ringcor, pinkycor, thumbco, handLms =  nooffingers(hands, rgbframe)
            if middle!=0 and ring!=0 and pinky!=0:
                if middlecor.y < reference:
                    pag.scroll(20)
                elif middlecor.y > reference:
                    pag.scroll(-20)
            cv.rectangle(capturedown, (frameReduction, frameReduction), (camwidth - frameReduction, camheight - frameReduction),(255, 0, 255), 2)
            if handLms:
                for i in handLms:
                    mpDraw.draw_landmarks(capturedown, i, mpHands.HAND_CONNECTIONS,mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())
            cv.imshow("frame",capturedown)
            cv.waitKey(1)
            if middle == 0 and ring == 0 and pinky == 0:
                cv.destroyAllWindows()
    
    #for right click
    if index == 1 and middle == 1 and ring == 1:
        pag.click(button="right")
        cv.destroyAllWindows()
        while middle!=0 and ring!=0:
            #waits until the fingers other than index are removed
            _, capturedown = webcam.read()
            rgbframe = cv.cvtColor(capturedown,cv.COLOR_BGR2RGB)
            index, middle, ring, pinky, thumb, indexcor, middlecor, ringcor, pinkycor, thumbco, handLms =  nooffingers(hands, rgbframe)
            cv.rectangle(capturedown, (frameReduction, frameReduction), (camwidth - frameReduction, camheight - frameReduction),(255, 0, 255), 2)
            if handLms:
                for i in handLms:
                    mpDraw.draw_landmarks(capturedown, i, mpHands.HAND_CONNECTIONS,mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())
            cv.imshow("frame",capturedown)
            cv.waitKey(1)
            if middle ==0 and ring ==0:
               cv.destroyAllWindows()
            
    #for left click and double-click        
    if index == 1 and middle == 1:
        pag.click()
        countdown = 0
        cv.destroyAllWindows()
        while middle!=0:
            #waits until the fingers other than index are removed,
            #and if middle finger is held up long enough, perfroms double click
            if countdown>50:
                pag.doubleClick()
                countdown=0
            _, capturedown = webcam.read()
            capturedown = capturedown.copy()
            rgbframe = cv.cvtColor(capturedown,cv.COLOR_BGR2RGB)
            index, middle, ring, pinky, thumb, indexcor, middlecor, ringcor, pinkycor, thumbco, handLms =  nooffingers(hands, rgbframe)
            countdown+=1
            cv.rectangle(capturedown, (frameReduction, frameReduction), (camwidth - frameReduction, camheight - frameReduction),(255, 0, 255), 2)
            if handLms:
                for i in handLms:
                    mpDraw.draw_landmarks(capturedown, i, mpHands.HAND_CONNECTIONS,mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())
            cv.imshow("frame",capturedown)
            cv.waitKey(1)
            if middle ==0:
                cv.destroyAllWindows()
            
    if handLms:
        for i in handLms:
            mpDraw.draw_landmarks(frame, i, mpHands.HAND_CONNECTIONS,mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())
    cv.rectangle(frame, (frameReduction, frameReduction), (camwidth - frameReduction, camheight - frameReduction),(255, 0, 255), 2)
    cv.putText(frame, str(int(fps)), (100, 100),cv.FONT_HERSHEY_SIMPLEX, 3, (100, 255, 0), 3, cv.LINE_AA)
    cv.putText(frame, str(index)+str(middle)+str(ring)+str(pinky)+str(thumb), (300, 100),cv.FONT_HERSHEY_SIMPLEX, 3, (100, 255, 0), 3, cv.LINE_AA)
    cv.imshow("frame",frame)
    
    prev_time = new_time
    
    if cv.waitKey(1)==ord('q'):
        cv.destroyAllWindows()
        break
    
#webcam.release_camera()



AttributeError: function 'free' not found